# Probability of Collision

This notebook presents a tool easily change a close approach geometry and assess its impact in the probability of collision.

## Setup

In [ ]:
import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from ostk.physics.coordinate import Frame
from ostk.physics.coordinate import Position
from ostk.physics.coordinate import Velocity
from ostk.physics.time import DateTime
from ostk.physics.time import Instant
from ostk.physics.time import Scale
from ostk.physics.unit import Derived

from ostk.astrodynamics.conjunction import CloseApproach
from ostk.astrodynamics.conjunction import ProbabilityOfCollisionAlgorithm
from ostk.astrodynamics.conjunction.hard_body import Spherical
from ostk.astrodynamics.conjunction.probability_of_collision_algorithm import Alfano2005
from ostk.astrodynamics.trajectory import LocalOrbitalFrameFactory
from ostk.astrodynamics.trajectory import State
from ostk.astrodynamics.trajectory.state.coordinate_subset import CartesianPosition
from ostk.astrodynamics.uncertainty import Covariance

## Close Approach

Define the time of closest approach (TCA) as well as the covariance matrices aspect ratios in RTN:

In [ ]:
tca: Instant = Instant.date_time(DateTime(2026, 10, 13, 15, 49, 0, 177), Scale.UTC)

# Position standard deviations [m] in RTN, for both objects
rtn_position_sigmas_m: np.ndarray = np.array([10.0, 100.0, 10.0])

object_1_state: State = State(
    tca,
    Position.meters(np.array([146.854845, 4463.520885, 5254.762166]) * 1e3, Frame.ITRF()),
    Velocity.meters_per_second(np.array([1.651218241, -5.736627232, 4.818385973]) * 1e3, Frame.ITRF()),
)

algorithm: Alfano2005 = Alfano2005()

## Utility functions

Define some utility functions to run the analysis.

In [ ]:
def generate_close_approach(
    sigma_scaling_factor: float,
    miss_distance_r_m: float,
    miss_distance_t_m: float,
    miss_distance_n_m: float,
    relative_velocity_angle_deg: float,
) -> CloseApproach:
    rtn_frame_factory: LocalOrbitalFrameFactory = LocalOrbitalFrameFactory.QSW(Frame.GCRF())

    rtn_frame: Frame = rtn_frame_factory.generate_frame(object_1_state)

    # 1000.0 m/s is a "high relative velocity encounter"
    relative_speed_m_s: float = 1000.0
    relative_velocity_angle_rad: float = np.deg2rad(relative_velocity_angle_deg)

    # Object 2 state relative to Object 1, in the RTN frame of Object 1: head-on along -T (0 deg), from the side along
    # N (90 deg)
    relative_state: State = State(
        tca,
        Position.meters(np.array([miss_distance_r_m, miss_distance_t_m, miss_distance_n_m]), rtn_frame),
        Velocity.meters_per_second(
            np.array(
                [
                    0.0,
                    -relative_speed_m_s * np.cos(relative_velocity_angle_rad),
                    relative_speed_m_s * np.sin(relative_velocity_angle_rad),
                ]
            ),
            rtn_frame,
        ),
    )

    object_2_state: State = relative_state.in_frame(Frame.GCRF())

    object_1_state_with_covariance, object_2_state_with_covariance = [
        state.with_covariance(
            Covariance(
                instant=tca,
                coordinates=np.diag(np.square(rtn_position_sigmas_m * sigma_scaling_factor)),
                frame=rtn_frame_factory.generate_frame(state),
                coordinate_subsets=[CartesianPosition.default()],
            )
        )
        for state in [object_1_state, object_2_state]
    ]

    return CloseApproach(
        object_1_state=object_1_state_with_covariance,
        object_2_state=object_2_state_with_covariance,
    )


# Encounter plane background color for each probability region (white otherwise)
region_background_colors: dict[ProbabilityOfCollisionAlgorithm.ProbabilityRegion, str] = {
    ProbabilityOfCollisionAlgorithm.ProbabilityRegion.Robust: "#dcf2e3",
    ProbabilityOfCollisionAlgorithm.ProbabilityRegion.Diluted: "#f8dede",
}

# Region plot sigma scaling factors, relative to the close approach covariances
region_sigma_scaling_factors: np.ndarray = np.logspace(-1.0, 1.0, num=100)


def plot_close_approach(close_approach: CloseApproach, combined_hard_body_radius_m: float) -> None:
    # Both objects share the same spherical hard body, of half the combined hard body radius
    hard_body: Spherical = Spherical(radius=combined_hard_body_radius_m / 2.0)

    unsatisfied_assumptions: list[str] = algorithm.identify_unsatisfied_assumptions(
        close_approach=close_approach,
    )

    if unsatisfied_assumptions:
        raise RuntimeError(f"{algorithm.get_name()} is not applicable: {unsatisfied_assumptions}")

    analysis: ProbabilityOfCollisionAlgorithm.Analysis = algorithm.compute_probability_analysis(
        close_approach=close_approach,
        hard_body_1=hard_body,
        hard_body_2=hard_body,
    )

    # Miss distance and combined position uncertainty in the rotated encounter frame, whose x-axis (resp. y-axis) is
    # along the major (resp. minor) principal axis of the combined position covariance
    rotated_encounter_frame_parameters: CloseApproach.RotatedEncounterFrameParameters = (
        close_approach.compute_rotated_encounter_frame_parameters(
            encounter_frame=close_approach.compute_encounter_frame(),
        )
    )

    x_m: float = float(rotated_encounter_frame_parameters.x_m.in_meters())
    y_m: float = float(rotated_encounter_frame_parameters.y_m.in_meters())
    sigma_x: float = float(rotated_encounter_frame_parameters.sigma_x.in_meters())
    sigma_y: float = float(rotated_encounter_frame_parameters.sigma_y.in_meters())

    miss_distance_rtn_m: list[float] = [
        float(component.in_meters())
        for component in close_approach.compute_miss_distance_components_in_local_orbital_frame(
            LocalOrbitalFrameFactory.QSW(Frame.GCRF())
        )
    ]

    # Relative velocity angle in the RTN frame of Object 1, from -T (0 deg when head-on) towards N
    relative_velocity_rtn_m_s: np.ndarray = (
        close_approach.get_object_2_state()
        .in_frame(LocalOrbitalFrameFactory.QSW(Frame.GCRF()).generate_frame(close_approach.get_object_1_state()))
        .get_velocity()
        .get_coordinates()
    )
    relative_velocity_angle_deg: float = float(
        np.rad2deg(np.arctan2(relative_velocity_rtn_m_s[2], -relative_velocity_rtn_m_s[1]))
    )

    # 1-, 2- and 3-sigma ellipses of the combined covariance (aligned with the rotated encounter frame axes), centered on
    # Object 2, and combined hard body, centered on Object 1
    angles: np.ndarray = np.linspace(0.0, 2.0 * np.pi, num=200)
    unit_circle: np.ndarray = np.array([np.cos(angles), np.sin(angles)])

    # Sigma level, outline color and fill color, from the largest ellipse (drawn first, below the others)
    sigma_ellipse_styles: list[tuple[int, str, str]] = [
        (3, "#e8913a", "rgba(255, 196, 140, 0.5)"),  # Light orange
        (2, "#d4b200", "rgba(255, 240, 150, 0.5)"),  # Light yellow
        (1, "#8c8c8c", "rgba(210, 210, 210, 0.5)"),  # Light grey
    ]

    ellipses: dict[int, np.ndarray] = {
        sigma_level: np.array([[x_m], [y_m]]) + sigma_level * np.array([[sigma_x], [sigma_y]]) * unit_circle
        for sigma_level, _, _ in sigma_ellipse_styles
    }

    combined_hard_body: np.ndarray = combined_hard_body_radius_m * unit_circle

    # Analysis table and encounter plane
    rows: dict[str, str] = {
        "Algorithm": str(algorithm.get_name()),
        "Combined Hard Body Radius [m]": f"{combined_hard_body_radius_m:.3g}",
        # Rounded to the millimeter to drop numerical noise (adding 0.0 turns -0.0 into 0.0)
        "Miss Distance R, T, N [m]": ", ".join(f"{round(component, 3) + 0.0:.3g}" for component in miss_distance_rtn_m),
        "Miss Distance [m]": f"{float(close_approach.compute_miss_distance().in_meters()):.3f}",
        "Relative Velocity Angle [deg]": f"{relative_velocity_angle_deg:.1f}",
        "Rotated Encounter Frame x_m, y_m [m]": f"{x_m:.3g}, {y_m:.3g}",
        "Rotated Encounter Frame sigma_x, sigma_y [m]": f"{sigma_x:.3g}, {sigma_y:.3g}",
        "Relative Speed [m/s]": f"{float(close_approach.compute_relative_speed().in_unit(Derived.Unit.meter_per_second())):.1f}",
        "Nominal Pc": f"{float(analysis.probability_of_collision):.3e}",
        "Maximum Pc": f"{float(analysis.maximum_probability_of_collision):.3e}",
        "Region": analysis.region.name,
        "Maximum Pc Scaling Factor": f"{float(analysis.maximum_probability_of_collision_scaling_factor):.3g}",
        "Maximum Pc Sigma Scaling Factor": f"{float(analysis.maximum_probability_of_collision_sigma_scaling_factor):.3g}",
    }

    encounter_figure: go.Figure = make_subplots(
        rows=2,
        cols=1,
        specs=[[{"type": "table"}], [{"type": "xy"}]],
        row_heights=[0.35, 0.65],
        vertical_spacing=0.08,
    )

    encounter_figure.add_trace(
        go.Table(
            header={"values": ["", "Value"], "align": "left"},
            cells={"values": [list(rows.keys()), list(rows.values())], "align": "left"},
        ),
        row=1,
        col=1,
    )

    for trace in [
        go.Scatter(
            x=combined_hard_body[0],
            y=combined_hard_body[1],
            mode="lines",
            line={"color": "#000000", "width": 2},
            fill="toself",
            fillcolor="#d9d9d9",
            hoverinfo="skip",
        ),
        *[
            go.Scatter(
                x=ellipses[sigma_level][0],
                y=ellipses[sigma_level][1],
                mode="lines",
                line={"color": line_color, "width": 2},
                fill="toself",
                fillcolor=fill_color,
                hoverinfo="skip",
            )
            for sigma_level, line_color, fill_color in sigma_ellipse_styles
        ],
        go.Scatter(
            x=[x_m],
            y=[y_m],
            mode="markers",
            marker={"color": "#2a78d6", "size": 10},
            hovertemplate="x: %{x:.1f} m<br>y: %{y:.1f} m<extra></extra>",
        ),
        go.Scatter(
            x=[0.0],
            y=[0.0],
            mode="markers",
            marker={"color": "#52514e", "size": 10, "symbol": "x"},
            hoverinfo="skip",
        ),
    ]:
        encounter_figure.add_trace(trace, row=2, col=1)

    # Labels inside the plot: the ellipse and hard body labels above their topmost point, the object labels below their
    # markers
    for x, y, text, yanchor, yshift in [
        *[
            (x_m, y_m + sigma_level * sigma_y, f"{sigma_level}-sigma combined covariance", "bottom", 4)
            for sigma_level, _, _ in sigma_ellipse_styles
        ],
        (0.0, combined_hard_body_radius_m, "Combined hard body", "bottom", 4),
        (x_m, y_m, "Object 2 relative position", "top", -8),
        (0.0, 0.0, "Object 1", "top", -8),
    ]:
        encounter_figure.add_annotation(
            x=x,
            y=y,
            text=text,
            showarrow=False,
            yanchor=yanchor,
            yshift=yshift,
            font={"color": "#52514e"},
            row=2,
            col=1,
        )

    encounter_figure.update_xaxes(title="Rotated encounter frame x [m]", row=2, col=1)
    encounter_figure.update_yaxes(title="Rotated encounter frame y [m]", scaleanchor="x", row=2, col=1)

    encounter_figure.update_layout(
        title="Probability of collision analysis and rotated encounter plane",
        height=900,
        template="plotly_white",
        showlegend=False,
        plot_bgcolor=region_background_colors.get(analysis.region, "#ffffff"),
    )

    # Region
    probabilities_of_collision: list[float] = [
        float(
            algorithm.compute_probability_of_collision(
                close_approach=close_approach.scale(
                    scale_factor_1=pow(sigma_scaling_factor, 2),
                    scale_factor_2=pow(sigma_scaling_factor, 2),
                ),
                hard_body_1=hard_body,
                hard_body_2=hard_body,
            )
        )
        for sigma_scaling_factor in region_sigma_scaling_factors
    ]

    region_figure: go.Figure = go.Figure(
        go.Scatter(
            x=region_sigma_scaling_factors,
            y=probabilities_of_collision,
            mode="lines+markers",
            line={"color": "#2a78d6", "width": 2},
            marker={"size": 8},
            hovertemplate="Scaling factor: %{x:.2g}<br>Pc: %{y:.3e}<extra></extra>",
        )
    )

    maximum_probability_of_collision_sigma_scaling_factor: float = float(
        analysis.maximum_probability_of_collision_sigma_scaling_factor
    )

    region_figure.add_vrect(
        x0=region_sigma_scaling_factors[0],
        x1=maximum_probability_of_collision_sigma_scaling_factor,
        fillcolor="#dcf2e3",
        line_width=0,
        layer="below",
    )
    region_figure.add_vrect(
        x0=maximum_probability_of_collision_sigma_scaling_factor,
        x1=region_sigma_scaling_factors[-1],
        fillcolor="#f8dede",
        line_width=0,
        layer="below",
    )

    # Label each region at its center (in log10 units, as for the line annotations)
    for region_name, region_lower_bound, region_upper_bound in [
        ("Robust", region_sigma_scaling_factors[0], maximum_probability_of_collision_sigma_scaling_factor),
        ("Diluted", maximum_probability_of_collision_sigma_scaling_factor, region_sigma_scaling_factors[-1]),
    ]:
        region_figure.add_annotation(
            x=(np.log10(region_lower_bound) + np.log10(region_upper_bound)) / 2.0,
            y=0.0,
            yref="paper",
            yanchor="bottom",
            yshift=8,
            showarrow=False,
            text=region_name,
            font={"color": "#52514e"},
        )

    region_figure.add_vline(x=1.0, line={"color": "#52514e", "width": 1})
    region_figure.add_vline(
        x=maximum_probability_of_collision_sigma_scaling_factor,
        line={"color": "#52514e", "width": 1, "dash": "dot"},
    )

    # Annotations are positioned in log10 units on a log axis (unlike shapes), hence added separately from the lines
    region_figure.add_annotation(
        x=np.log10(1.0),
        y=1.0,
        yref="paper",
        yanchor="bottom",
        showarrow=False,
        text=f"Nominal Pc: {float(analysis.probability_of_collision):.3e}",
    )
    region_figure.add_annotation(
        x=np.log10(maximum_probability_of_collision_sigma_scaling_factor),
        y=1.0,
        yref="paper",
        yanchor="bottom",
        yshift=20,
        showarrow=False,
        text=f"Maximum Pc: {float(analysis.maximum_probability_of_collision):.3e}",
    )

    region_figure.update_layout(
        title="Probability of collision regions",
        xaxis_title="Covariance sigma scaling factor",
        xaxis_type="log",
        xaxis_range=[np.log10(region_sigma_scaling_factors[0]), np.log10(region_sigma_scaling_factors[-1])],
        yaxis_title="Probability of collision",
        yaxis_tickformat=".1e",
        template="plotly_white",
        margin={"t": 120},
        showlegend=False,
    )

    encounter_figure.show()
    region_figure.show()

## Analysis

Run a probability of collision analysis, allowing parameters to be modified in real time:

In [ ]:
slider_options: dict = {
    "continuous_update": False,
    "layout": {"width": "420px"},
    "style": {"description_width": "160px"},
}

# Miss distance slider options: 0, then logarithmic steps from 0.1 m to 1000.0 m (10 steps per decade)
miss_distance_options_m: np.ndarray = np.array([0.0, *np.logspace(-1.0, 3.0, num=41)])

sliders: dict[str, widgets.Widget] = {
    "sigma_scaling_factor": widgets.FloatLogSlider(
        value=1.0,
        base=10.0,
        min=-1.0,
        max=2.0,
        step=0.1,
        description="Sigma scaling factor",
        **slider_options,
    ),
    **{
        f"miss_distance_{component.lower()}_m": widgets.SelectionSlider(
            options=[(f"{option_m:.3g}", float(option_m)) for option_m in miss_distance_options_m],
            value=float(miss_distance_options_m[np.argmin(np.abs(miss_distance_options_m - initial_value_m))]),
            description=f"Miss distance {component} [m]",
            **slider_options,
        )
        for component, initial_value_m in [("R", 50.0), ("T", 50.0), ("N", 50.0)]
    },
    "relative_velocity_angle_deg": widgets.FloatSlider(
        value=45.0,
        min=0.0,
        max=90.0,
        step=1.0,
        description="Relative velocity angle [deg]",
        **slider_options,
    ),
    "combined_hard_body_radius_m": widgets.FloatLogSlider(
        value=20.0,
        base=10.0,
        min=-1.0,
        max=2.0,
        step=0.1,
        description="Combined hard body radius [m]",
        **slider_options,
    ),
}

# Table and plots on the left, sliders on the right
output: widgets.Output = widgets.interactive_output(
    lambda combined_hard_body_radius_m, **close_approach_inputs: plot_close_approach(
        close_approach=generate_close_approach(**close_approach_inputs),
        combined_hard_body_radius_m=combined_hard_body_radius_m,
    ),
    sliders,
)
output.layout = widgets.Layout(flex="1 1 auto", min_width="0")

display(
    widgets.HBox(
        [
            output,
            widgets.VBox(
                [widgets.HTML("<h3>Geometry</h3>"), *sliders.values()],
                layout={"flex": "0 0 auto"},
            ),
        ],
        layout={"align_items": "flex-start"},
    )
)